# Tutorial: GEN-02 GAN Training on a Synthetic Mixture

- Module: 11 Generative Models
- Topic: adversarial learning and alternating optimization
- Estimated runtime: 2-3 minutes on CPU
- Prerequisites: probability distributions, binary cross-entropy, stochastic gradient descent, PyTorch basics
- Expected outputs: discriminator/generator diagnostics and a plot comparing real and generated samples
    


## Learning goals

- Implement a minimal GAN training loop with separate generator and discriminator steps.
- Observe how adversarial losses differ from maximum-likelihood style objectives.
- Visualize mode coverage on a synthetic two-dimensional data distribution.
    


In [ ]:
from __future__ import annotations

import math

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(11)
np.random.seed(11)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device
    


## Synthetic mixture dataset

GANs are easiest to debug when the target distribution is visible.
We use eight Gaussian clusters arranged on a ring, which makes mode dropping visually obvious.
    


In [ ]:
def make_ring_mixture(n_samples: int = 4096, radius: float = 2.5, noise: float = 0.08) -> np.ndarray:
    angles = np.linspace(0, 2 * math.pi, 8, endpoint=False)
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    assignments = np.random.randint(0, len(centers), size=n_samples)
    samples = centers[assignments] + noise * np.random.randn(n_samples, 2)
    return samples.astype(np.float32)


real_data = make_ring_mixture()
loader = DataLoader(TensorDataset(torch.tensor(real_data)), batch_size=256, shuffle=True)
real_data[:3]
    


## Generator and discriminator

The discriminator outputs a logit for the event "this point came from the data".
The generator maps Gaussian noise `z ~ N(0, I)` into candidate samples in `R^2`.
    


In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim: int = 2, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


generator = Generator().to(device)
discriminator = Discriminator().to(device)
g_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-3, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-3, betas=(0.5, 0.999))
criterion = nn.BCEWithLogitsLoss()
    


## Alternating adversarial updates

For each minibatch we first train the discriminator on real and fake points, then train the generator to make fake points score as real.
This is not likelihood maximization; it is a game between two networks.
    


In [ ]:
def sample_noise(batch_size: int, noise_dim: int = 2) -> torch.Tensor:
    return torch.randn(batch_size, noise_dim, device=device)


history = []
for epoch in range(1, 201):
    d_running = 0.0
    g_running = 0.0
    acc_real = 0.0
    acc_fake = 0.0
    batches = 0

    for (xb,) in loader:
        xb = xb.to(device)
        batch_size = xb.size(0)
        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        # Discriminator step
        fake_points = generator(sample_noise(batch_size)).detach()
        real_logits = discriminator(xb)
        fake_logits = discriminator(fake_points)
        d_loss = criterion(real_logits, real_labels) + criterion(fake_logits, fake_labels)
        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Generator step
        generated = generator(sample_noise(batch_size))
        fool_logits = discriminator(generated)
        g_loss = criterion(fool_logits, real_labels)
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        d_running += d_loss.item()
        g_running += g_loss.item()
        acc_real += (torch.sigmoid(real_logits) > 0.5).float().mean().item()
        acc_fake += (torch.sigmoid(fake_logits) < 0.5).float().mean().item()
        batches += 1

    if epoch % 20 == 0 or epoch == 1:
        summary = {
            'epoch': epoch,
            'd_loss': d_running / batches,
            'g_loss': g_running / batches,
            'real_acc': acc_real / batches,
            'fake_acc': acc_fake / batches,
        }
        history.append(summary)
        print(
            f"epoch={epoch:03d} d_loss={summary['d_loss']:.3f} g_loss={summary['g_loss']:.3f} "
            f"real_acc={summary['real_acc']:.2f} fake_acc={summary['fake_acc']:.2f}"
        )

history[-1]
    


## Diagnostics

Balanced training is the difficult part of GANs.
If the discriminator becomes perfect, the generator loses a useful gradient; if the generator wins too early, the discriminator stops providing a meaningful signal.
    


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot([h['epoch'] for h in history], [h['d_loss'] for h in history], marker='o', label='D loss')
ax[0].plot([h['epoch'] for h in history], [h['g_loss'] for h in history], marker='s', label='G loss')
ax[0].set_title('Adversarial losses')
ax[0].set_xlabel('Epoch')
ax[0].legend()

ax[1].plot([h['epoch'] for h in history], [h['real_acc'] for h in history], marker='o', label='Real accuracy')
ax[1].plot([h['epoch'] for h in history], [h['fake_acc'] for h in history], marker='s', label='Fake accuracy')
ax[1].set_title('Discriminator accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylim(0.0, 1.05)
ax[1].legend()
plt.tight_layout()
fig
    


## Compare real and generated points

A successful run places synthetic points near all eight modes, but GAN training offers no explicit likelihood and no guaranteed latent geometry.
Mode dropping appears as missing clusters in the generator output.
    


In [ ]:
generator.eval()
with torch.no_grad():
    fake_samples = generator(sample_noise(1500)).cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(real_data[:, 0], real_data[:, 1], s=8, alpha=0.25, label='real')
ax.scatter(fake_samples[:, 0], fake_samples[:, 1], s=8, alpha=0.55, label='generated')
ax.set_title('Ring mixture: real vs generated samples')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend()
ax.axis('equal')
fig
    


## Summary

The GAN objective can yield sharp samples without normalized likelihoods, but optimization stability is the core tradeoff.
This two-dimensional example makes the minimax game visible before moving to high-dimensional image models.
    
